# E28 - Stateful ONLINE detectors (CUSUM / Page-Hinkley / BOCPD)

**The real paradigm shift.** The champion sees a *stateless window snapshot* at each step.
These detectors instead carry **state across the whole online stream** - the natural shape
of the problem. They are **unsupervised** (no training): each emits a per-step score
directly, which we rank with TS-AUC and compare to the trained champion.

- `cusum`         - two-sided CUSUM of standardized residuals vs history (mean/var).
- `page_hinkley`  - Page-Hinkley cumulative-deviation test.
- `bocpd`         - Bayesian Online Changepoint Detection (Gaussian, constant hazard),
                    score = P(a changepoint has occurred since online start).

Hardened bar: to *replace* the champion a detector must beat ~0.5795; even a *weaker* score
that is strongly **complementary** could later be added as a feature (noted for follow-up).
`SMOKE=1` for a quick check.

In [ ]:
import os, math
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

from _overnight import load_pool, select_picks, evaluate, paired, SMOKE, N_JOBS
from _exp_fast import split_series, ts_auc_from
print("SMOKE =", SMOKE)

from _feat_lib import _feat

BASE_KEYS = ("spec_entropy",)   # base15 builder
SCALES = (60, 130, 250)         # champion multiscale
WINDOW = 250


def _subwin(w, s):
    return w if s >= len(w) else w[-s:]


def champion_feat(w, ctx):
    out = []
    for s in SCALES:
        out.extend(_feat(_subwin(w, s), ctx, keys=BASE_KEYS))
    return out

def verdict(md_, t):
    if md_ > 0 and t >= 2.5:
        return "SHIP(hard)"
    if md_ > 0 and t >= 2.0:
        return "marginal(HOLD)"
    if md_ > 0 and t >= 1.5:
        return "weak"
    return "noise"


In [ ]:
EPS = 1e-12


def cusum_scores(x_h, x_on, k=0.5, H=8.0):
    mu = float(np.mean(x_h)) if len(x_h) else 0.0
    sd = float(np.std(x_h)) or 1.0
    sp = sm = 0.0
    out = np.zeros(len(x_on))
    for i, x in enumerate(x_on):
        z = (x - mu) / sd
        sp = max(0.0, sp + z - k)
        sm = max(0.0, sm - z - k)
        out[i] = math.tanh(max(sp, sm) / H)
    return out


def page_hinkley_scores(x_h, x_on, delta=0.005, H=50.0):
    mu = float(np.mean(x_h)) if len(x_h) else 0.0
    sd = float(np.std(x_h)) or 1.0
    m = 0.0
    Mmin = 0.0
    out = np.zeros(len(x_on))
    for i, x in enumerate(x_on):
        z = (x - mu) / sd
        m += z - delta
        Mmin = min(Mmin, m)
        ph = m - Mmin
        # also track negative direction
        out[i] = math.tanh(abs(ph) / H)
    return out


def bocpd_scores(x_h, x_on, lam=250.0, Rmax=300):
    x_h = np.asarray(x_h, dtype=np.float64)
    mu0 = float(x_h.mean()) if len(x_h) else 0.0
    var_x = float(x_h.var()) + EPS
    var0 = var_x
    hz = 1.0 / lam
    R = np.array([1.0])
    counts = np.array([0.0])
    sums = np.array([0.0])
    n = len(x_on)
    out = np.zeros(n)
    for t in range(n):
        x = float(x_on[t])
        post_prec = 1.0 / var0 + counts / var_x
        post_mean = (mu0 / var0 + sums / var_x) / post_prec
        pred_var = var_x + 1.0 / post_prec
        pred = np.exp(-0.5 * (x - post_mean) ** 2 / pred_var) / np.sqrt(2 * np.pi * pred_var)
        growth = R * pred * (1.0 - hz)
        cp = float(np.sum(R * pred * hz))
        newR = np.empty(len(R) + 1)
        newR[0] = cp
        newR[1:] = growth
        counts = np.concatenate([[0.0], counts + 1.0])
        sums = np.concatenate([[0.0], sums + x])
        s = newR.sum() + EPS
        newR = newR / s
        if len(newR) > Rmax:
            newR = newR[:Rmax]; counts = counts[:Rmax]; sums = sums[:Rmax]
            newR = newR / (newR.sum() + EPS)
        R = newR
        out[t] = 1.0 - float(R[min(t, len(R) - 1)])
    return out


DETECTORS = {"cusum": cusum_scores, "page_hinkley": page_hinkley_scores, "bocpd": bocpd_scores}


In [ ]:
def _detector_series(det_fn, sid, x_h, x_on, onl_index):
    sc = det_fn(x_h, x_on)
    midx = pd.MultiIndex.from_product([[sid], onl_index], names=["id", "time"])
    return pd.Series(sc, index=midx)


def eval_detector(det_fn, X, y_val, folds):
    aucs = []
    for f in folds:
        jobs = []
        for sid in f:
            x_h, onl = split_series(X.loc[sid])
            jobs.append((sid, x_h.to_numpy(dtype=np.float64) if hasattr(x_h, "to_numpy") else np.asarray(x_h),
                         onl.to_numpy(dtype=np.float64), onl.index))
        with threadpool_limits(limits=1):
            series = Parallel(n_jobs=N_JOBS, backend="loky")(
                delayed(_detector_series)(det_fn, sid, x_h, x_on, oi)
                for (sid, x_h, x_on, oi) in jobs
            )
        aucs.append(ts_auc_from(pd.concat(series).to_frame("prediction"), y_val))
    return np.asarray(aucs)


X, yi, y_val, train_ids, folds = load_pool()
picks = select_picks(X, train_ids)
base_aucs, nf = evaluate(champion_feat, WINDOW, X, yi, y_val, picks, folds)
print(f"champion (trained) nf={nf} mean={base_aucs.mean():.4f}\n")

for name, fn in DETECTORS.items():
    a = eval_detector(fn, X, y_val, folds)
    md_, sd, t, _ = paired(a, base_aucs)
    tag = verdict(md_, t) if md_ > 0 else "below champion"
    print(f"{name:>13}  mean={a.mean():.4f}  vs champion delta={md_:+.4f}  t={t:+.2f}  -> {tag}")


## Interpretation

These are *unsupervised* and untrained, so beating a trained 45-feature GBT outright is a
high bar. Two useful outcomes:

1. A detector **beats** the champion -> huge: a stateful paradigm genuinely wins; build it out.
2. A detector is weaker but **complementary** (uncorrelated errors) -> its per-step score is a
   strong candidate *feature* for the champion (a stateful summary the window snapshot can't
   see). Follow-up: append `bocpd`/`cusum` score as a feature and validate at t>=2.5.

If all three track below the champion with correlated ranking, the trained window model
already captures their signal and the stateful lever is closed.